# LAB 08 — Tokenization, Special Tokens, Attention Mask, MLM
**Pretrained Transformers and BERT Fine-Tuning** · Môn NLP & ứng dụng, HK I 2026

Notebook này gồm: Calculation 2 (tokenization `"The cats are playing."`), kiểm tra `tokens.keys()` / `input_ids` / `attention_mask`,
subword tokenization, kiểm chứng Calculation 1 (MLM loss) bằng BERT thật, contextual embedding, và phân bố độ dài token của IMDb
(liên quan Exp 4). Các ô đánh dấu ✍️ là phần diễn giải.

In [1]:
import numpy as np, pandas as pd, torch, matplotlib
matplotlib.use("Agg")
from transformers import AutoTokenizer, AutoModel, BertForMaskedLM
import warnings, logging; warnings.filterwarnings("ignore"); logging.getLogger("transformers").setLevel(logging.ERROR)

MODEL = "bert-base-uncased"
tok = AutoTokenizer.from_pretrained(MODEL)
print("tokenizer:", type(tok).__name__, "| vocab size:", tok.vocab_size, "| model_max_length:", tok.model_max_length)
print("special tokens:", tok.special_tokens_map)
print({t: tok.convert_tokens_to_ids(t) for t in ["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"]})

tokenizer: BertTokenizer | vocab size: 30522 | model_max_length: 512
special tokens: {'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}
{'[PAD]': 0, '[UNK]': 100, '[CLS]': 101, '[SEP]': 102, '[MASK]': 103}


## 1. Calculation 2 — Tokenize `"The cats are playing."`

In [2]:
text = "The cats are playing."
enc = tok(text, return_tensors="pt")
print("keys:", list(enc.keys()))
tokens = tok.convert_ids_to_tokens(enc["input_ids"][0])
ids = enc["input_ids"][0].tolist()
mask = enc["attention_mask"][0].tolist()
special = [t for t in tokens if t in tok.all_special_tokens]
words = text.split()
print("tokens        :", tokens)
print("token IDs     :", ids)
print("special tokens:", special)
print("attention mask:", mask)
print()
n_words_ws = len(words)                       # tách bằng khoảng trắng
n_words_punct = len(tok.backend_tokenizer.pre_tokenizer.pre_tokenize_str(text))  # tách cả dấu câu (pre-tokenizer của BERT)
n_tokens_all = len(tokens)
n_tokens_content = n_tokens_all - len(special)
n_subword = sum(t.startswith("##") for t in tokens)
print(pd.DataFrame({"đại lượng": ["words (tách khoảng trắng)", "words+punct (pre-tokenizer BERT)", "tokens (kể cả special)", "tokens (không tính special)", "subword pieces (bắt đầu bằng ##)"],
                    "số lượng": [n_words_ws, n_words_punct, n_tokens_all, n_tokens_content, n_subword]}).to_string(index=False))

keys: ['input_ids', 'token_type_ids', 'attention_mask']
tokens        : ['[CLS]', 'the', 'cats', 'are', 'playing', '.', '[SEP]']
token IDs     : [101, 1996, 8870, 2024, 2652, 1012, 102]
special tokens: ['[CLS]', '[SEP]']
attention mask: [1, 1, 1, 1, 1, 1, 1]

                       đại lượng  số lượng
       words (tách khoảng trắng)         4
words+punct (pre-tokenizer BERT)         5
          tokens (kể cả special)         7
     tokens (không tính special)         5
subword pieces (bắt đầu bằng ##)         0


In [3]:
# Với câu này không có từ nào bị tách subword. Thử các từ khác để thấy WordPiece hoạt động:
for w in ["unbelievable", "tokenization", "playing", "cats", "unhappiness", "transformers", "antidisestablishmentarianism", "COVID-19", "Phạm Ngọc Hải", "😀"]:
    print(f"{w!r:36s} -> {tok.tokenize(w)}")

'unbelievable'                       -> ['unbelievable']
'tokenization'                       -> ['token', '##ization']
'playing'                            -> ['playing']
'cats'                               -> ['cats']
'unhappiness'                        -> ['un', '##ha', '##pp', '##iness']
'transformers'                       -> ['transformers']
'antidisestablishmentarianism'       -> ['anti', '##dis', '##est', '##ab', '##lish', '##ment', '##arian', '##ism']
'COVID-19'                           -> ['co', '##vid', '-', '19']
'Phạm Ngọc Hải'                      -> ['ph', '##am', 'ngo', '##c', 'hai']
'😀'                                  -> ['[UNK]']


✍️ **Nhận xét (Calculation 2).**
- Tokenizer `bert-base-uncased` (WordPiece, lowercase) cho `['[CLS]', 'the', 'cats', 'are', 'playing', '.', '[SEP]']`, IDs `[101, 1996, 8870, 2024, 2652, 1012, 102]`.
- Special tokens: `[CLS]` (id 101) đầu câu và `[SEP]` (id 102) cuối câu; tokenizer tự thêm. Với 1 câu không padding, attention mask toàn 1 (`[1,1,1,1,1,1,1]`).
- Số lượng: 4 word (tách khoảng trắng) / 5 đơn vị nếu tính dấu `.` / 7 token (5 nội dung + 2 special) / **0 subword piece**. Nghĩa là `cats` và `playing` đều là token nguyên vẹn trong vocab 30,522 — không phải từ nào cũng bị tách. Subword chỉ xuất hiện với từ hiếm / dài / ít gặp: `tokenization → token ##ization`, `unhappiness → un ##ha ##pp ##iness`, `COVID-19 → co ##vid - 19`, tên riêng tiếng Việt bị vỡ vụn thành `ph ##am ngo ##c hai` (mất dấu vì uncased + bỏ dấu). Lưu ý `unbelievable` vẫn là **một** token trong vocab này, trái với ví dụ minh họa trong đề (`un + believe + able`) — đề cũng nói "một tokenizer cụ thể có thể tạo ra subword khác". Emoji thành `[UNK]`.
- Đề không yêu cầu đoán trước tokenization; mình chạy rồi mới ghi, nên đây là quan sát chứ không phải dự đoán.

## 2. `tokens.keys()`, `input_ids`, `attention_mask`, padding và truncation

In [4]:
batch = ["The cat eats fish.", "The cats are playing in the garden with a very long, long tail.", "Great!"]
tokens_b = tok(batch, padding=True, truncation=True, max_length=12, return_tensors="pt")
print("tokens.keys():", list(tokens_b.keys()))
for i, s in enumerate(batch):
    print(f"\n[{i}] {s}")
    print("   tokens :", tok.convert_ids_to_tokens(tokens_b['input_ids'][i]))
    print("   ids    :", tokens_b["input_ids"][i].tolist())
    print("   mask   :", tokens_b["attention_mask"][i].tolist())
print("\nshape input_ids:", tuple(tokens_b["input_ids"].shape), "(batch, T_max) — T_max <= max_length=12")

tokens.keys(): ['input_ids', 'token_type_ids', 'attention_mask']

[0] The cat eats fish.
   tokens : ['[CLS]', 'the', 'cat', 'eats', 'fish', '.', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']
   ids    : [101, 1996, 4937, 20323, 3869, 1012, 102, 0, 0, 0, 0, 0]
   mask   : [1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0]

[1] The cats are playing in the garden with a very long, long tail.
   tokens : ['[CLS]', 'the', 'cats', 'are', 'playing', 'in', 'the', 'garden', 'with', 'a', 'very', '[SEP]']
   ids    : [101, 1996, 8870, 2024, 2652, 1999, 1996, 3871, 2007, 1037, 2200, 102]
   mask   : [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]

[2] Great!
   tokens : ['[CLS]', 'great', '!', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']
   ids    : [101, 2307, 999, 102, 0, 0, 0, 0, 0, 0, 0, 0]
   mask   : [1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0]

shape input_ids: (3, 12) (batch, T_max) — T_max <= max_length=12


In [5]:
# Cặp câu: token_type_ids phân biệt segment A / B
pair = tok("sentence A", "sentence B", return_tensors="pt")
print(tok.convert_ids_to_tokens(pair["input_ids"][0]))
print("token_type_ids:", pair["token_type_ids"][0].tolist())

['[CLS]', 'sentence', 'a', '[SEP]', 'sentence', 'b', '[SEP]']
token_type_ids: [0, 0, 0, 0, 1, 1, 1]


In [6]:
# Vì sao cần attention mask: so sánh [CLS] của "Great!" khi KHÔNG padding và khi có padding (mask đúng / mask sai)
bert = AutoModel.from_pretrained(MODEL).eval()
one = tok("Great!", return_tensors="pt")
pad = tok("Great!", padding="max_length", max_length=12, return_tensors="pt")
cos = torch.nn.functional.cosine_similarity
with torch.no_grad():
    h_no_pad = bert(**one).last_hidden_state[0, 0]
    h_pad_ok = bert(input_ids=pad["input_ids"], attention_mask=pad["attention_mask"]).last_hidden_state[0, 0]
    h_pad_bad = bert(input_ids=pad["input_ids"], attention_mask=torch.ones_like(pad["attention_mask"])).last_hidden_state[0, 0]
print("max |không pad - pad có mask đúng| =", (h_no_pad - h_pad_ok).abs().max().item())
print("cos(không pad, pad có mask đúng)   =", cos(h_no_pad, h_pad_ok, dim=0).item())
print("cos(không pad, pad bị coi là token) =", cos(h_no_pad, h_pad_bad, dim=0).item(), " <- padding làm biến đổi biểu diễn")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

max |không pad - pad có mask đúng| = 9.5367431640625e-07
cos(không pad, pad có mask đúng)   = 1.0
cos(không pad, pad bị coi là token) = 0.6028716564178467  <- padding làm biến đổi biểu diễn


✍️ **Nhận xét.** `input_ids` là chỉ số token trong vocab (đưa vào embedding); `attention_mask` có 1 cho token thật, 0 cho `[PAD]`; `token_type_ids` chỉ có ý nghĩa khi có cặp câu.
Khi có padding mà mask đúng, `[CLS]` gần như trùng với trường hợp không padding (sai khác nhỏ do số học float); nếu coi `[PAD]` là token thật thì biểu diễn bị lệch (cosine < 1, xem output). Đây là bằng chứng
cho thấy mask là cần thiết chứ không phải thủ tục. Với `max_length=12`, câu dài bị cắt và `[SEP]` vẫn được giữ ở cuối.

## 3. Kiểm chứng Calculation 1 (MLM) bằng BERT thật

In [7]:
# (a) Tính loss theo đề: P(eats)=0.70, 0.95, 0.01  (đã tính tay trong calculations.md, đây chỉ là đối chiếu)
for p in [0.70, 0.95, 0.01]:
    print(f"P(eats)={p:<5} -> L = -ln(p) = {-np.log(p):.4f}")

P(eats)=0.7   -> L = -ln(p) = 0.3567
P(eats)=0.95  -> L = -ln(p) = 0.0513
P(eats)=0.01  -> L = -ln(p) = 4.6052


In [8]:
# (b) BERT thật dự đoán token bị che trong "The cat [MASK] fish."
mlm = BertForMaskedLM.from_pretrained(MODEL).eval()
s = "The cat [MASK] fish."
e = tok(s, return_tensors="pt")
pos = (e["input_ids"][0] == tok.mask_token_id).nonzero().item()
with torch.no_grad():
    logits = mlm(**e).logits[0, pos]
p = torch.softmax(logits, -1)
top = torch.topk(p, 8)
print("Top-8 cho vị trí [MASK]:")
for pr, i in zip(top.values, top.indices):
    print(f"  {tok.convert_ids_to_tokens(i.item()):10s} {pr.item():.4f}")
for w in ["eats", "likes", "dog"]:
    wid = tok.convert_tokens_to_ids(w)
    print(f"P({w}) = {p[wid].item():.4f} -> loss nếu đáp án là '{w}' = {-torch.log(p[wid]).item():.4f}")
y = torch.tensor([tok.convert_tokens_to_ids("eats")])
print("F.cross_entropy (đáp án 'eats') =", torch.nn.functional.cross_entropy(logits[None], y).item())

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Top-8 cho vị trí [MASK]:
  and        0.5909
  -          0.0706
  .          0.0401
  or         0.0209
  is         0.0198
  ,          0.0180
  the        0.0172
  of         0.0115
P(eats) = 0.0035 -> loss nếu đáp án là 'eats' = 5.6549
P(likes) = 0.0003 -> loss nếu đáp án là 'likes' = 8.1638
P(dog) = 0.0000 -> loss nếu đáp án là 'dog' = 10.3004
F.cross_entropy (đáp án 'eats') = 5.654855728149414


✍️ **Nhận xét.** `-ln(0.70) = 0.3567`; nếu 0.95 thì 0.0513, nếu 0.01 thì 4.6052 (xem `calculations.md`). Model BERT thật (chưa fine-tune, chỉ có MLM head) dự đoán top-1 cho `The cat [MASK] fish.` là `and` (P≈0.59) và chỉ gán P(`eats`)≈0.0035, tương ứng loss ≈ 5.65 — rất khác con số giả định 0.70 trong đề. Hợp lý vì câu ngắn, thiếu ngữ cảnh: `The cat and fish.` cũng là câu hợp lệ, MLM chỉ thấy 2 từ xung quanh. Vậy 0.70 trong đề chỉ là số minh họa, và loss thấp hay cao phụ thuộc hoàn toàn vào xác suất model gán cho token thật.
`F.cross_entropy(logits, y)` trùng với `-log softmax(logits)[y]`, nên MLM loss thực chất là cross-entropy phân loại trên toàn bộ vocab (30,522 lớp) tại các vị trí bị che.

## 4. Contextual embedding vs static embedding

In [9]:
def word_vec(model, sentence, word, layer=-1):
    e = tok(sentence, return_tensors="pt")
    toks = tok.convert_ids_to_tokens(e["input_ids"][0])
    with torch.no_grad():
        hs = model(**e, output_hidden_states=True).hidden_states[layer][0]
    return hs[toks.index(word)]
s1, s2, s3 = "He sat on the bank of the river.", "She deposited money at the bank.", "They walked along the river bank."
a, b, c = (word_vec(bert, s, "bank") for s in (s1, s2, s3))
print("Static embedding 'bank': một vector duy nhất -> cos = 1.0 giữa mọi cặp câu")
print(f"Contextual cos(river-bank 1, money-bank)  = {cos(a, b, dim=0).item():.3f}")
print(f"Contextual cos(river-bank 1, river-bank 2) = {cos(a, c, dim=0).item():.3f}")

Static embedding 'bank': một vector duy nhất -> cos = 1.0 giữa mọi cặp câu


Contextual cos(river-bank 1, money-bank)  = 0.518
Contextual cos(river-bank 1, river-bank 2) = 0.821


✍️ **Nhận xét.** Embedding tĩnh của `bank` là một vector duy nhất; sau khi qua encoder, vector của `bank` phụ thuộc ngữ cảnh: hai câu cùng nghĩa "bờ sông" nên gần nhau hơn so với cặp "bờ sông" vs "ngân hàng" (xem cosine). Đây là ý nghĩa của "contextual embedding" trong chuỗi `Tokenizer → Embedding → Transformer`. (Chỉ 3 câu nên đây là minh họa, không phải đo lường có ý nghĩa thống kê.)

## 5. Độ dài token IMDb (liên quan Exp 4: max sequence length)

In [10]:
import sys; sys.path.insert(0, ".")
import pretrained_classification as pc
train_df, val_df, test_df = pc.load_imdb_splits()
lens = np.array([len(x) for x in tok(train_df["text"].tolist(), truncation=False)["input_ids"]])
print(pd.Series(lens).describe().round(1).to_string())
for T in (64, 128, 256, 512):
    print(f"T={T:3d}: {100*(lens > T).mean():5.1f}% review bị cắt (>T token, tính cả [CLS]/[SEP]); giữ lại {100*np.minimum(lens, T).sum()/lens.sum():.1f}% tổng số token")
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(6, 3.2))
ax.hist(np.clip(lens, 0, 1200), bins=60, color="#4C78A8")
for T in (64, 128, 256, 512): ax.axvline(T, color="#E45756", ls="--", lw=1)
ax.set_xlabel("số token WordPiece / review (cắt ở 1200)"); ax.set_ylabel("số review"); ax.set_title("Phân bố độ dài token (IMDb train)")
fig.tight_layout(); fig.savefig("token_length_distribution.png", dpi=150)

count    20000.0
mean       296.8
std        220.9
min         13.0
25%        161.0
50%        222.0
75%        360.0
max       3047.0
T= 64:  97.8% review bị cắt (>T token, tính cả [CLS]/[SEP]); giữ lại 21.5% tổng số token
T=128:  87.8% review bị cắt (>T token, tính cả [CLS]/[SEP]); giữ lại 41.5% tổng số token
T=256:  41.1% review bị cắt (>T token, tính cả [CLS]/[SEP]); giữ lại 68.5% tổng số token
T=512:  13.2% review bị cắt (>T token, tính cả [CLS]/[SEP]); giữ lại 89.1% tổng số token


✍️ **Nhận xét.** IMDb có review dài hơn nhiều so với 128 token; trung bình ~297 token (median 222). Ở T=64 có 97.8% review bị cắt (chỉ giữ 21.5% tổng token), T=128: 87.8% (giữ 41.5%), T=256: 41.1% (giữ 68.5%), T=512: 13.2%. Do đó khi so sánh T∈{64,128,256} (Exp 4), tăng T nghĩa là cho model nhìn thêm nội dung (phần cuối review thường chứa kết luận), chứ không chỉ là tăng chi phí.